# Angelic Search 

Search using angelic semantics (is a hierarchical search), where the agent chooses the implementation of the HLA's. <br>
The algorithms input is: problem, hierarchy and initialPlan
-  problem is of type Problem 
-  hierarchy is a dictionary consisting of all the actions. 
-  initialPlan is an approximate description(optimistic and pessimistic) of the agents choices for the implementation. <br>
   initialPlan contains a sequence of HLA's with angelic semantics

In [1]:
from planning import * 
import inspect

The Angelic search algorithm consists of three parts. 
-  Search using angelic semantics
-  Decompose
-  a search in the space of refinements, in a similar way with hierarchical search

### Searching using angelic semantics
-  Find the reachable set (optimistic and pessimistic) of the sequence of angelic HLA in initialPlan
  -    If the optimistic reachable set doesn't intersect the goal, then there is no solution
  -    If the pessimistic reachable set intersects the goal, then we call decompose, in order to find the sequence of actions that lead us to the goal. 
  -    If the optimistic reachable set intersects the goal, but the pessimistic doesn't we do some further refinements, in order to see if there is a sequence of actions that achieves the goal. 
  
### Search in space of refinements
-  Create a search tree, that has root the action and children it's refinements
-  Extend frontier by adding each refinement, so that we keep looping till we find all primitive actions
-  If we achieve that we return the path of the solution (search tree), else there is no solution and we return None.

  


In [2]:
print(inspect.getsource(RealWorldPlanningProblem.angelic_search))

    def angelic_search(self, hierarchy, initial_plan):
        """
        [Figure 11.8]
        A hierarchical planning algorithm that uses angelic semantics to identify and
        commit to high-level plans that work while avoiding high-level plans that don’t.
        The predicate MAKING-PROGRESS checks to make sure that we aren’t stuck in an infinite regression
        of refinements.
        At top level, call ANGELIC-SEARCH with [Act] as the initialPlan.

        InitialPlan contains a sequence of HLA's with angelic semantics

        The possible effects of an angelic HLA in initialPlan are:
        ~ : effect remove
        $+: effect possibly add
        $-: effect possibly remove
        $$: possibly add or remove
        """
        frontier = deque(initial_plan)
        while True:
            if not frontier:
                return None
            plan = frontier.popleft()  # sequence of HLA/Angelic HLA's
            opt_reachable_set = RealWorldPlanningProblem.reach_opt


### Decompose 
-  Finds recursively the sequence of states and actions that lead us from initial state to goal.
-  For each of the above actions we find their refinements,if they are not primitive, by calling the angelic_search function. 
   If there are not refinements return None
   


In [3]:
print(inspect.getsource(RealWorldPlanningProblem.decompose))

    def decompose(hierarchy, plan, s_f, reachable_set):
        solution = []
        i = max(reachable_set.keys())
        while plan.action_pes:
            action = plan.action_pes.pop()
            if i == 0:
                return solution
            s_i = RealWorldPlanningProblem.find_previous_state(s_f, reachable_set, i, action)
            problem = RealWorldPlanningProblem(s_i, s_f, plan.action)
            angelic_call = RealWorldPlanningProblem.angelic_search(problem, hierarchy,
                                                                   [AngelicNode(s_i, Node(None), [action], [action])])
            if angelic_call:
                for x in angelic_call:
                    solution.insert(0, x)
            else:
                return None
            s_f = s_i
            i -= 1
        return solution



## Example

Suppose that somebody wants to get to the airport. 
The possible ways to do so is either get a taxi, or drive to the airport. <br>
Those two actions have some preconditions and some effects. 
If you get the taxi, you need to have cash, whereas if you drive you need to have a car. <br>
Thus we define the following hierarchy of possible actions.

##### hierarchy

In [4]:
library = {
        'HLA': ['Go(Home,SFO)', 'Go(Home,SFO)', 'Drive(Home, SFOLongTermParking)', 'Shuttle(SFOLongTermParking, SFO)', 'Taxi(Home, SFO)'],
        'steps': [['Drive(Home, SFOLongTermParking)', 'Shuttle(SFOLongTermParking, SFO)'], ['Taxi(Home, SFO)'], [], [], []],
        'precond': [['At(Home) & Have(Car)'], ['At(Home)'], ['At(Home) & Have(Car)'], ['At(SFOLongTermParking)'], ['At(Home)']],
        'effect': [['At(SFO) & ~At(Home)'], ['At(SFO) & ~At(Home) & ~Have(Cash)'], ['At(SFOLongTermParking) & ~At(Home)'], ['At(SFO) & ~At(LongTermParking)'], ['At(SFO) & ~At(Home) & ~Have(Cash)']] }




the possible actions are the following:

In [5]:
go_SFO = HLA('Go(Home,SFO)', precond='At(Home)', effect='At(SFO) & ~At(Home)')
taxi_SFO = HLA('Taxi(Home,SFO)', precond='At(Home)', effect='At(SFO) & ~At(Home) & ~Have(Cash)')
drive_SFOLongTermParking = HLA('Drive(Home, SFOLongTermParking)', 'At(Home) & Have(Car)','At(SFOLongTermParking) & ~At(Home)' )
shuttle_SFO = HLA('Shuttle(SFOLongTermParking, SFO)', 'At(SFOLongTermParking)', 'At(SFO) & ~At(LongTermParking)')

Suppose that (our preconditionds are that) we are Home and we have cash and car and  our goal is to get to SFO and maintain our cash, and our possible actions are the above. <br>
##### Then our problem is: 

In [6]:
prob = RealWorldPlanningProblem('At(Home) & Have(Cash) & Have(Car)', 'At(SFO) & Have(Cash)', [go_SFO, taxi_SFO, drive_SFOLongTermParking,shuttle_SFO])

An agent gives us some approximate information about the plan we will follow: <br>
(initialPlan is an Angelic Node, where: 
-    state is the initial state of the problem, 
-    parent is None 
-    action: is a list of actions (Angelic HLA's) with the optimistic estimators of effects and 
-    action_pes: is a list of actions (Angelic HLA's) with the pessimistic approximations of the effects
##### InitialPlan

In [7]:
angelic_opt_description = AngelicHLA('Go(Home, SFO)', precond = 'At(Home)', effect ='$+At(SFO) & $-At(Home)' ) 
angelic_pes_description = AngelicHLA('Go(Home, SFO)', precond = 'At(Home)', effect ='$+At(SFO) & ~At(Home)' )

initialPlan = [AngelicNode(prob.initial, None, [angelic_opt_description], [angelic_pes_description])] 


We want to find the optimistic and pessimistic reachable set of initialPlan when applied to the problem:
##### Optimistic/Pessimistic reachable set

In [8]:
opt_reachable_set = RealWorldPlanningProblem.reach_opt(prob.initial, initialPlan[0])
pes_reachable_set = RealWorldPlanningProblem.reach_pes(prob.initial, initialPlan[0])

# reachable sets 
print([x for y in opt_reachable_set.keys() for x in opt_reachable_set[y]], '\n') # type: ignore
print([x for y in pes_reachable_set.keys() for x in pes_reachable_set[y]]) # type: ignore


[[At(Home), Have(Cash), Have(Car)], [Have(Cash), Have(Car), At(SFO), NotAt(Home)], [Have(Cash), Have(Car), NotAt(Home)], [At(Home), Have(Cash), Have(Car), At(SFO)], [At(Home), Have(Cash), Have(Car)]] 

[[At(Home), Have(Cash), Have(Car)], [Have(Cash), Have(Car), At(SFO), NotAt(Home)], [Have(Cash), Have(Car), NotAt(Home)]]


##### Refinements

In [9]:
#TODO

Run the angelic search

In [10]:
plan= RealWorldPlanningProblem.angelic_search(prob, library, initialPlan)
print (plan, '\n')
print ([x.__dict__ for x in plan]) # type: ignore

[Drive(Home, SFOLongTermParking), Shuttle(SFOLongTermParking, SFO)] 

[{'name': 'Drive', 'args': (Home, SFOLongTermParking), 'precond': [At(Home), Have(Car)], 'effect': [At(SFOLongTermParking), NotAt(Home)], 'domain': None, 'duration': 0, 'consumes': {}, 'uses': {}, 'completed': False}, {'name': 'Shuttle', 'args': (SFOLongTermParking, SFO), 'precond': [At(SFOLongTermParking)], 'effect': [At(SFO), NotAt(LongTermParking)], 'domain': None, 'duration': 0, 'consumes': {}, 'uses': {}, 'completed': False}]


## Example 2

In [20]:
library_2 = {
        'HLA': ['Go(Home,SFO)', 'Go(Home,SFO)', 'Bus(Home, MetroStop)', 'Metro(MetroStop, SFO)' , 'Metro(MetroStop, SFO)', 'Metro1(MetroStop, SFO)', 'Metro2(MetroStop, SFO)'  ,'Taxi(Home, SFO)'],
        'steps': [['Bus(Home, MetroStop)', 'Metro(MetroStop, SFO)'], ['Taxi(Home, SFO)'], [], ['Metro1(MetroStop, SFO)'], ['Metro2(MetroStop, SFO)'],[],[],[]],
        'precond': [['At(Home)'], ['At(Home)'], ['At(Home)'], ['At(MetroStop)'], ['At(MetroStop)'],['At(MetroStop)'], ['At(MetroStop)'] ,['At(Home) & Have(Cash)']],
        'effect': [['At(SFO) & ~At(Home)'], ['At(SFO) & ~At(Home) & ~Have(Cash)'], ['At(MetroStop) & ~At(Home)'], ['At(SFO) & ~At(MetroStop)'], ['At(SFO) & ~At(MetroStop)'], ['At(SFO) & ~At(MetroStop)'] , ['At(SFO) & ~At(MetroStop)'] ,['At(SFO) & ~At(Home) & ~Have(Cash)']] 
        }

In [23]:
# only prob, library and initplan are needed
plan_2 = RealWorldPlanningProblem.angelic_search(prob, library_2, initialPlan)
print(plan_2)


[Bus(Home, MetroStop), Metro1(MetroStop, SFO)]


## Example 3 

Sometimes there is no plan that achieves the goal!

In [4]:
library_3 = {
        'HLA': ['Shuttle(SFOLongTermParking, SFO)', 'Go(Home, SFOLongTermParking)', 'Taxi(Home, SFOLongTermParking)', 'Drive(Home, SFOLongTermParking)', 'Drive(SFOLongTermParking, Home)', 'Get(Cash)', 'Go(Home, ATM)'],
        'steps': [['Get(Cash)', 'Go(Home, SFOLongTermParking)'], ['Taxi(Home, SFOLongTermParking)'], [], [], [], ['Drive(SFOLongTermParking, Home)', 'Go(Home, ATM)'], []],
        'precond': [['At(SFOLongTermParking)'], ['At(Home)'], ['At(Home) & Have(Cash)'], ['At(Home)'],  ['At(SFOLongTermParking)'], ['At(SFOLongTermParking)'], ['At(Home)']],
        'effect': [['At(SFO)'], ['At(SFO)'], ['At(SFOLongTermParking) & ~Have(Cash)'], ['At(SFOLongTermParking)'] ,['At(Home) & ~At(SFOLongTermParking)'], ['At(Home) & Have(Cash)'], ['Have(Cash)'] ]
        }


In [5]:
# individual HLAs
shuttle_SFO = HLA('Shuttle(SFOLongTermParking, SFO)', 'Have(Cash) & At(SFOLongTermParking)', 'At(SFO)')
prob_3 = RealWorldPlanningProblem('At(SFOLongTermParking) & Have(Cash)', 'At(SFO) & Have(Cash)', [shuttle_SFO])

# optimistic/pessimistic descriptions for init
angelic_opt_description = AngelicHLA('Shuttle(SFOLongTermParking, SFO)', precond = 'At(SFOLongTermParking)', effect ='$+At(SFO) & $-At(SFOLongTermParking)' ) 
angelic_pes_description = AngelicHLA('Shuttle(SFOLongTermParking, SFO)', precond = 'At(SFOLongTermParking)', effect ='$+At(SFO) & ~At(SFOLongTermParking)' ) 

# initial Plan
initialPlan_3 = [AngelicNode(prob_3.initial, None, [angelic_opt_description], [angelic_pes_description])] 

In [11]:
plan_3 = RealWorldPlanningProblem.angelic_search(prob_3, library_3, initialPlan_3)
print(plan_3)

NameError: name 'prob_3' is not defined